In [1]:
# Basic data handling
import pandas as pd
import numpy as np
from imblearn.combine import SMOTEENN

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn utilities
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# Metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

# Optuna for hyperparameter tuning
import optuna
from optuna.integration import OptunaSearchCV

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import optuna

# Warnings
import warnings
warnings.filterwarnings('ignore')


c:\Users\Louis Bernal\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('bank-additional-full.csv', sep=';')

In [3]:
df.drop_duplicates(inplace=True)

In [4]:
df_object = df.select_dtypes(include = 'object')

for col in df_object:
    df[col] = LabelEncoder().fit_transform(df[col])

In [5]:
x = df.drop('y', axis = 1)
y = df['y']

In [6]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y)

In [7]:
smote_enn = SMOTEENN(random_state =42)
x_train, y_train = smote_enn.fit_resample(x_train, y_train)

In [8]:
sc = RobustScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [9]:
input_dim = x_train.shape[1]

In [10]:
def build_tf_model(trial, input_dim):

    model = Sequential()

    # Tune number of hidden layers
    n_layers = trial.suggest_int("n_layers", 1, 3)

    for i in range(n_layers):
        units = trial.suggest_int(f"units_l{i}", 16, 128)
        dropout = trial.suggest_float(f"dropout_l{i}", 0.1, 0.5)

        if i == 0:
            model.add(Dense(units, activation="relu", input_dim=input_dim))
        else:
            model.add(Dense(units, activation="relu"))

        model.add(BatchNormalization())
        model.add(Dropout(dropout))

    # Output layer
    model.add(Dense(1, activation="sigmoid"))

    # Tune learning rate
    lr = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["AUC"]   # better for imbalanced data
    )

    return model


In [11]:
def objective(trial):

    # Proper train/validation split
    X_tr, X_val, y_tr, y_val = train_test_split(
        x_train, y_train,
        test_size=0.2,
        stratify=y_train,
        random_state=42
    )

    model = build_tf_model(trial, input_dim=X_tr.shape[1])

    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=0
    )

    # Train
    model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=0
    )

    # Predict validation
    val_preds = model.predict(X_val, verbose=0)

    # Compute ROC-AUC safely
    auc = roc_auc_score(y_val, val_preds)

    return auc


In [12]:
study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=20)

print("Best Trial Parameters:")
print(study.best_trial.params)


[I 2026-02-11 09:54:13,873] A new study created in memory with name: no-name-24e994d8-144c-4f37-8168-71c2a7379098
[I 2026-02-11 09:54:29,061] Trial 0 finished with value: 0.9869020225467194 and parameters: {'n_layers': 3, 'units_l0': 41, 'dropout_l0': 0.10319787056137658, 'units_l1': 25, 'dropout_l1': 0.19777649416406265, 'units_l2': 71, 'dropout_l2': 0.44007882681909394, 'learning_rate': 0.00013170064913037096, 'batch_size': 256}. Best is trial 0 with value: 0.9869020225467194.
[I 2026-02-11 09:55:16,085] Trial 1 finished with value: 0.9913228268459084 and parameters: {'n_layers': 2, 'units_l0': 56, 'dropout_l0': 0.2823872823367565, 'units_l1': 116, 'dropout_l1': 0.38397537239996604, 'learning_rate': 0.0002574364471991081, 'batch_size': 64}. Best is trial 1 with value: 0.9913228268459084.
[I 2026-02-11 09:55:36,170] Trial 2 finished with value: 0.9881368878015526 and parameters: {'n_layers': 3, 'units_l0': 65, 'dropout_l0': 0.1612852528254193, 'units_l1': 19, 'dropout_l1': 0.272948870

Best Trial Parameters:
{'n_layers': 2, 'units_l0': 128, 'dropout_l0': 0.22460958912919438, 'units_l1': 127, 'dropout_l1': 0.49593583327644464, 'learning_rate': 0.0011867040766300827, 'batch_size': 128}


In [13]:
best_params = study.best_trial.params

final_model = build_tf_model(
    optuna.trial.FixedTrial(best_params),
    input_dim=x_train.shape[1]
)

early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

final_model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=best_params["batch_size"],
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/200
326/326 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - AUC: 0.9678 - loss: 0.2308 - val_AUC: 0.0000e+00 - val_loss: 0.1763
Epoch 2/200
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.9813 - loss: 0.1716 - val_AUC: 0.0000e+00 - val_loss: 0.1301
Epoch 3/200
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.9844 - loss: 0.1553 - val_AUC: 0.0000e+00 - val_loss: 0.1256
Epoch 4/200
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.9863 - loss: 0.1439 - val_AUC: 0.0000e+00 - val_loss: 0.0963
Epoch 5/200
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.9872 - loss: 0.1379 - val_AUC: 0.0000e+00 - val_loss: 0.0953
Epoch 6/200
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.9878 - loss: 0.1328 - val_AUC: 0.0000e+00 - val_loss: 0.0843
Epoch 7/200
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.9882 - loss: 0.1312 - val_AUC: 0.0000e+00 - val_loss: 0.0711
Epoch 8/200
326/326 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.9887 - loss: 0.1271 - val_AUC: 0.0000e+00 - val_loss: 0.0764
Epoch 9/200
326/

In [14]:
# Evaluation
tf_probs = final_model.predict(x_test)
tf_pred = (tf_probs > 0.5).astype(int)

print("Tuned TensorFlow ANN:\n")
print(classification_report(y_test, tf_pred))

258/258 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Tuned TensorFlow ANN:

              precision    recall  f1-score   support

           0       0.99      0.85      0.91      7308
           1       0.43      0.93      0.59       928

    accuracy                           0.86      8236
   macro avg       0.71      0.89      0.75      8236
weighted avg       0.93      0.86      0.88      8236



In [15]:
# %%
# Create models directory first
import os
os.makedirs('models', exist_ok=True)

# %%
# ============================================
# SAVE MODEL AND PREPROCESSING OBJECTS
# ============================================

import pickle
import json

# 1. Save the trained TensorFlow model
final_model.save('models/bank_ads_model.h5')
print("✓ Model saved to models/bank_ads_model.h5")

# 2. Save the scaler
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(sc, f)
print("✓ Scaler saved to models/scaler.pkl")

# 3. Save label encoders (re-fit and save)
label_encoders = {}
df_original = pd.read_csv('bank-additional-full.csv', sep=';')

for col in df_original.select_dtypes(include='object').columns:
    le = LabelEncoder()
    le.fit(df_original[col])
    label_encoders[col] = le

with open('models/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)
print("✓ Label encoders saved to models/label_encoders.pkl")

# 4. Save feature names (column order matters!)
feature_columns = list(x.columns)
with open('models/feature_columns.json', 'w') as f:
    json.dump(feature_columns, f)
print("✓ Feature columns saved to models/feature_columns.json")

# 5. Save best hyperparameters
with open('models/best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
print("✓ Best parameters saved to models/best_params.json")

# 6. Save model performance metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

metrics = {
    'accuracy': float(accuracy_score(y_test, tf_pred)),
    'precision': float(precision_score(y_test, tf_pred)),
    'recall': float(recall_score(y_test, tf_pred)),
    'f1_score': float(f1_score(y_test, tf_pred)),
    'roc_auc': float(roc_auc_score(y_test, tf_probs))
}

with open('models/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("✓ Metrics saved to models/metrics.json")

print("\n" + "="*50)
print("ALL FILES SAVED SUCCESSFULLY!")
print("="*50)
print("\nFiles created:")
print("  - models/bank_ads_model.h5")
print("  - models/scaler.pkl")
print("  - models/label_encoders.pkl")
print("  - models/feature_columns.json")
print("  - models/best_params.json")
print("  - models/metrics.json")

✓ Model saved to models/bank_ads_model.h5
✓ Scaler saved to models/scaler.pkl
✓ Label encoders saved to models/label_encoders.pkl
✓ Feature columns saved to models/feature_columns.json
✓ Best parameters saved to models/best_params.json
✓ Metrics saved to models/metrics.json

ALL FILES SAVED SUCCESSFULLY!

Files created:
  - models/bank_ads_model.h5
  - models/scaler.pkl
  - models/label_encoders.pkl
  - models/feature_columns.json
  - models/best_params.json
  - models/metrics.json


In [16]:
final_model.save('models/bank_ads_model.keras')